# Heart Disease Prediction — Deep Learning (PBL)

A fully-connected neural network (MLP) that predicts the presence of heart disease
from 13 clinical features of the **UCI Cleveland Heart Disease** dataset.

**Team:** Aedula Sricharan (23EG106C01), Akkinepally Nagasai (23EG106C02), Akshith Sai Nalla (23EG106C03), Vivek Reddy A (23EG106C04)  
**Course:** Deep Learning — Project-Based Learning

This notebook walks through the whole pipeline end to end: data → EDA →
preprocessing → model → training → evaluation → single-patient inference.

## 1. Setup & imports

In [ ]:
import sys, os
# Make the project importable and use the repo root as the working directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from src.data import load_raw, prepare_data, FEATURE_DESCRIPTIONS, CONTINUOUS
from src.model import HeartMLP

sns.set_theme(style='whitegrid')
torch.manual_seed(42); np.random.seed(42)
print('torch', torch.__version__)

## 2. Load the data

In [ ]:
df = load_raw('data/heart.csv')
print('shape:', df.shape)
df.head()

In [ ]:
print('Class balance (0 = no disease, 1 = disease):')
print(df['target'].value_counts().sort_index())
print('\nMissing values:', int(df.isnull().sum().sum()))

## 3. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
df['target'].value_counts().sort_index().plot(kind='bar', ax=ax, color=['#4C72B0','#C44E52'])
ax.set_xticklabels(['No disease','Disease'], rotation=0)
ax.set_title('Target distribution'); ax.set_ylabel('count')
plt.tight_layout(); plt.show()

In [ ]:
# Distribution of continuous features split by the target
fig, axes = plt.subplots(1, len(CONTINUOUS), figsize=(18,3))
for ax, col in zip(axes, CONTINUOUS):
    sns.kdeplot(data=df, x=col, hue='target', fill=True, common_norm=False, ax=ax, legend=False)
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9,7))
sns.heatmap(df.corr(numeric_only=True), annot=False, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Feature correlation'); plt.tight_layout(); plt.show()

## 4. Preprocessing

Continuous features are standardised, multi-class categoricals are one-hot
encoded, and binary flags are passed through — all inside one scikit-learn
`ColumnTransformer`. The split is stratified and deterministic (seed = 42).

In [ ]:
data = prepare_data(path='data/heart.csv', seed=42)
print('train:', data.X_train.shape)
print('val:  ', data.X_val.shape)
print('test: ', data.X_test.shape)
print('input dimension after preprocessing:', data.input_dim)

## 5. Model

A compact MLP with batch-norm and dropout — well suited to a small tabular dataset.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HeartMLP(data.input_dim, hidden_dims=(64, 32), dropout=0.3).to(device)
print(model)

## 6. Training

Adam + `BCEWithLogitsLoss`, with early stopping on the validation loss.

In [ ]:
def loader(X, y, bs, shuffle):
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=bs, shuffle=shuffle)

train_loader = loader(data.X_train, data.y_train, 32, True)
val_loader   = loader(data.X_val,   data.y_val,   32, False)

pos = data.y_train.sum(); neg = len(data.y_train) - pos
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([neg/max(pos,1)], device=device))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

@torch.no_grad()
def evaluate(loader):
    model.eval(); loss=0; correct=0; n=0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss += criterion(logits, yb).item()*len(yb)
        correct += ((torch.sigmoid(logits)>=0.5).float()==yb).sum().item(); n += len(yb)
    return loss/n, correct/n

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best, best_state, patience = float('inf'), None, 25
for epoch in range(1, 201):
    model.train()
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad(); loss = criterion(model(xb), yb); loss.backward(); optimizer.step()
    tl, ta = evaluate(train_loader); vl, va = evaluate(val_loader)
    for k,v in zip(history, [tl,vl,ta,va]): history[k].append(v)
    if vl < best - 1e-4:
        best, best_state, patience = vl, {k:v.cpu().clone() for k,v in model.state_dict().items()}, 25
    else:
        patience -= 1
    if epoch % 25 == 0 or epoch == 1:
        print(f'epoch {epoch:3d} | train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va:.3f}')
    if patience <= 0:
        print(f'early stop @ {epoch}'); break
model.load_state_dict(best_state)
print('best val loss:', round(best,4))

In [ ]:
ep = range(1, len(history['train_loss'])+1)
fig,(a1,a2)=plt.subplots(1,2,figsize=(10,4))
a1.plot(ep, history['train_loss'], label='train'); a1.plot(ep, history['val_loss'], label='val'); a1.set_title('Loss'); a1.legend()
a2.plot(ep, history['train_acc'], label='train'); a2.plot(ep, history['val_acc'], label='val'); a2.set_title('Accuracy'); a2.legend()
plt.tight_layout(); plt.show()

## 7. Evaluation on the held-out test set

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, roc_curve, classification_report)
model.eval()
with torch.no_grad():
    prob = torch.sigmoid(model(torch.from_numpy(data.X_test).to(device))).cpu().numpy()
pred = (prob >= 0.5).astype(int); true = data.y_test.astype(int)
print(f'Accuracy : {accuracy_score(true,pred):.3f}')
print(f'Precision: {precision_score(true,pred):.3f}')
print(f'Recall   : {recall_score(true,pred):.3f}')
print(f'F1       : {f1_score(true,pred):.3f}')
print(f'ROC-AUC  : {roc_auc_score(true,prob):.3f}')
print('\n', classification_report(true, pred, target_names=['No disease','Disease']))

In [ ]:
cm = confusion_matrix(true, pred)
fpr, tpr, _ = roc_curve(true, prob)
fig,(a1,a2)=plt.subplots(1,2,figsize=(10,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=a1,
            xticklabels=['No disease','Disease'], yticklabels=['No disease','Disease'])
a1.set_xlabel('Predicted'); a1.set_ylabel('Actual'); a1.set_title('Confusion matrix')
a2.plot(fpr, tpr, label=f'AUC={roc_auc_score(true,prob):.3f}'); a2.plot([0,1],[0,1],'--',color='grey')
a2.set_xlabel('FPR'); a2.set_ylabel('TPR'); a2.set_title('ROC curve'); a2.legend()
plt.tight_layout(); plt.show()

## 8. Single-patient inference (demo)

In [ ]:
from src.data import FEATURE_COLUMNS
sample = {'age':57,'sex':1,'cp':0,'trestbps':130,'chol':236,'fbs':0,'restecg':0,
          'thalach':174,'exang':0,'oldpeak':0.0,'slope':1,'ca':1,'thal':2}
row = pd.DataFrame([{c: sample[c] for c in FEATURE_COLUMNS}])
X = data.preprocessor.transform(row).astype('float32')
with torch.no_grad():
    p = torch.sigmoid(model(torch.from_numpy(X).to(device))).item()
print(f'P(disease) = {p:.3f}  ->  ' + ('Heart disease likely' if p>=0.5 else 'No heart disease'))

## 9. Conclusion

The MLP reaches strong accuracy and ROC-AUC on the held-out test set, showing that
a simple deep network with sound preprocessing and regularisation handles this
clinical tabular problem well. Natural next steps: k-fold cross-validation for a
tighter estimate, hyper-parameter search, and comparison against classical models
(logistic regression, random forest, gradient boosting).